In [2]:
%load_ext autoreload
%autoreload 2

import sys
import os

sys.path.append(os.path.abspath(".."))

from src.geometry import AirfoilGeometry
from src.physics import AeroSolver
import numpy as np
import optuna
import pandas as pd

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [3]:
# Fixed parameters
CHORD = 1 
ALPHA = 0
N_POINTS_DATASET = 100
N_TRIALS_PER_OPTI = 100
dataset = []

for i in range(N_POINTS_DATASET):
    v_target = np.random.uniform(50, 250)    # Velocity between 10 et 45 m/s
    alt_target = np.random.uniform(0, 12000) # Altitude between 0 et 2500 m
    study = optuna.create_study(direction="maximize")

    def objective(trial):
        m = trial.suggest_int("m", 0, 9)
        p = trial.suggest_int("p", 0, 9)
        t = trial.suggest_int("t", 1, 40)
        
        af = AirfoilGeometry(n_points=100)
        xu, yu, xl, yl = af.generate_naca4(m_int=m, p_int=p, t_int=t)
        x_coords, y_coords = af.get_coords_for_solver(xu, yu, xl, yl)
        
        solver = AeroSolver(altitude=alt_target, velocity=v_target, chord=CHORD)
        res = solver.solve(x_coords, y_coords, alpha=ALPHA)
        
        cl = res['cl'][0]
        cd = res['cd'][0]
        
        if cd <= 0: return 0
        
        return cl / cd

    optuna.logging.set_verbosity(optuna.logging.WARNING)

    study.optimize(objective, n_trials=N_TRIALS_PER_OPTI, show_progress_bar=False)
    
    best = study.best_params
    dataset.append({
        'velocity': v_target,
        'altitude': alt_target,
        'best_m': best['m'],
        'best_p': best['p'],
        'best_t': int(best['t']),
        'finesse_max': study.best_value,
        'naca_complet': f"{best['m']}{best['p']}{int(best['t']):02d}"
    })
    
    print(f"✅ Point {i+1}/{N_POINTS_DATASET} | V={v_target:.1f}m/s | Alt={alt_target:.0f}m -> NACA {dataset[-1]['naca_complet']}")

df_airfoil_ai = pd.DataFrame(dataset)
print("\nDataset ready")
display(df_airfoil_ai.head())

[I 2026-03-27 12:02:51,058] A new study created in memory with name: no-name-679c40b3-9cb6-4a11-b75d-c1f661472b6f


✅ Point 1/100 | V=146.5m/s | Alt=9767m -> NACA 9706
✅ Point 2/100 | V=122.1m/s | Alt=9312m -> NACA 8706
✅ Point 3/100 | V=101.3m/s | Alt=7893m -> NACA 9705
✅ Point 4/100 | V=80.2m/s | Alt=8290m -> NACA 8705
✅ Point 5/100 | V=117.2m/s | Alt=1534m -> NACA 9603
✅ Point 6/100 | V=139.5m/s | Alt=11259m -> NACA 8705
✅ Point 7/100 | V=237.2m/s | Alt=7015m -> NACA 3603
✅ Point 8/100 | V=146.0m/s | Alt=7282m -> NACA 9603
✅ Point 9/100 | V=172.6m/s | Alt=10030m -> NACA 9603
✅ Point 10/100 | V=136.1m/s | Alt=9721m -> NACA 9706
✅ Point 11/100 | V=149.8m/s | Alt=3166m -> NACA 9605
✅ Point 12/100 | V=154.5m/s | Alt=4824m -> NACA 9603
✅ Point 13/100 | V=144.0m/s | Alt=3454m -> NACA 9604
✅ Point 14/100 | V=249.7m/s | Alt=459m -> NACA 4604
✅ Point 15/100 | V=176.5m/s | Alt=7717m -> NACA 9604
✅ Point 16/100 | V=242.1m/s | Alt=4728m -> NACA 4604
✅ Point 17/100 | V=199.2m/s | Alt=1772m -> NACA 9605
✅ Point 18/100 | V=195.8m/s | Alt=1170m -> NACA 9605
✅ Point 19/100 | V=118.5m/s | Alt=4431m -> NACA 9604
✅ 

,velocity,altitude,best_m,best_p,best_t,finesse_max,naca_complet
0,146.476560,9766.798567,9,7,6,347.721204,9706
1,122.078430,9312.391134,8,7,6,335.461349,8706
2,101.279395,7893.209265,9,7,5,327.646356,9705
3,80.164651,8289.852628,8,7,5,309.457211,8705
4,117.223557,1534.123651,9,6,3,366.604520,9603


In [4]:
df_airfoil_ai.to_csv("../data/airfoil_optimization_results.csv", index=False)